# DTSC 2302 — Fantasy Football Prediction Project
### Group Project Starter Notebook

**Research question:** How accurately can a player's weekly fantasy football PPR points be predicted using prior performances, position, and offensive statistics?

This notebook follows the kickoff worksheet: research question, data identification, variable definitions, data quality assessment, analysis, modeling, and next steps. Run cells top-to-bottom. **Requires an internet connection when fetching NFLverse data.**

## 1. Project idea and research design

- **Unit of analysis:** one NFL player in one regular-season week.
- **Target (dependent variable):** `fantasy_points_ppr` in the player's next observed game/week.
- **Features (independent variables):** recent historical PPR points, prior-game passing/rushing/receiving yards, receptions, touchdowns, and the player's position.
- **Method:** supervised regression, comparing a simple rolling-average baseline, linear regression, and random forest.
- **Why it matters:** helps understand whether recent statistical performance can meaningfully predict future fantasy production; real lineup decisions also require injuries, matchups, and playing time not fully captured here.
- **Dataset options:** [nflverse](https://github.com/nflverse/nflverse-data) (used here); [Kaggle NFL datasets](https://www.kaggle.com/datasets) (alternative, check licensing, columns, and coverage).

**Limitations to investigate:** bye weeks and missed games; injury/usage shifts; inconsistent player participation; team changes; different league scoring rules; selection bias from filtering to fantasy positions. The model does not include pregame injury status, and therefore should not be treated as a complete lineup optimizer.

## 2. Install libraries (once per environment)

In [ ]:
%pip install -q nflreadpy pandas numpy matplotlib seaborn scikit-learn pyarrow

## 3. Import packages and download actual weekly NFL stats

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nflreadpy as nfl
from IPython.display import display

SEASONS = [2023, 2024, 2025]  # 2025 is held out for evaluation
POSITIONS = ["QB", "RB", "WR", "TE"]

raw = nfl.load_player_stats(seasons=SEASONS, summary_level="week").to_pandas()
print("Raw data shape:", raw.shape)
print("First columns:", raw.columns.tolist()[:35])
display(raw.head())

## 4. Clean data and inspect missing values

We use published PPR fantasy points instead of reconstructing league-specific scoring. We keep regular-season records for QB/RB/WR/TE, use player ID to identify players across seasons, and make sure weeks are numeric.

In [ ]:
required = ["player_id", "player_display_name", "position", "season", "week", "fantasy_points_ppr"]
missing_required = [c for c in required if c not in raw.columns]
if missing_required:
    raise ValueError(f"Expected columns missing from NFLverse dataset: {missing_required}")

df = raw.copy()
if "season_type" in df.columns:
    df = df[df["season_type"].astype(str).str.upper().eq("REG")].copy()
df = df[df["position"].isin(POSITIONS)].copy()
df["week"] = pd.to_numeric(df["week"], errors="coerce")
df["season"] = pd.to_numeric(df["season"], errors="coerce")
df["fantasy_points_ppr"] = pd.to_numeric(df["fantasy_points_ppr"], errors="coerce")
df = df.dropna(subset=required).copy()
df = df.drop_duplicates(subset=["player_id", "season", "week"])
df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

print("Cleaned rows:", len(df))
print("Duplicate player-season-week entries:", df.duplicated(["player_id", "season", "week"]).sum())
print("Missing data in relevant columns:")
display(df[required].isna().sum().to_frame("missing"))
display(df.groupby(["season", "position"]).size().unstack(fill_value=0))

## 5. Explore fantasy scoring by position

In [ ]:
summary = df.groupby("position")["fantasy_points_ppr"].agg(["count", "mean", "median", "std"]).round(2)
display(summary)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="position", y="fantasy_points_ppr", order=POSITIONS, showfliers=False)
plt.title("Distribution of Weekly PPR Fantasy Points by Position")
plt.xlabel("Position")
plt.ylabel("PPR Fantasy Points")
plt.tight_layout()
plt.show()

## 6. Explore historical leaders

In [ ]:
totals = (df.groupby(["season", "player_id", "player_display_name", "position"], as_index=False)
          ["fantasy_points_ppr"].sum())
print("Top 10 players by total PPR points in 2025 (among records in this dataset):")
display(totals[totals["season"] == 2025].nlargest(10, "fantasy_points_ppr")
        [["player_display_name", "position", "fantasy_points_ppr"]])

## 7. Build **pregame** prediction features

This is the most important methodological step: feature values come from **prior games only**. We shift each player's stats by one observed game, then calculate averages of the preceding three games. A player needs at least three earlier recorded games *in that season* to be included. All feature construction happens separately for each player and season, preventing history from crossing season boundaries. A missed game or bye week is not explicitly modeled: these are previous *appearances*, not necessarily consecutive calendar weeks.

In [ ]:
candidate_stats = [
    "fantasy_points_ppr", "passing_yards", "rushing_yards",
    "receiving_yards", "receptions", "passing_tds",
    "rushing_tds", "receiving_tds"
]
feature_cols = []
group_cols = ["player_id", "season"]

def rolling_prior_three(series):
    return series.shift(1).rolling(window=3, min_periods=3).mean()

for stat in candidate_stats:
    if stat not in df.columns:
        print(f"Skipping unavailable field: {stat}")
        continue
    df[stat] = pd.to_numeric(df[stat], errors="coerce")
    new_name = "prior3_" + stat
    df[new_name] = (df.groupby(group_cols)[stat]
                    .transform(rolling_prior_three))
    feature_cols.append(new_name)

# Require three earlier appearances, and remove rows with unavailable past-stat features.
df["prior_appearances"] = df.groupby(group_cols).cumcount()
model_df = df[df["prior_appearances"] >= 3].dropna(
    subset=["fantasy_points_ppr"] + feature_cols
).copy()

if model_df.empty:
    raise ValueError("No model rows found. Inspect missing values and dataset columns.")

print("Model rows:", len(model_df))
print("Numeric feature columns:", feature_cols)
display(model_df[["player_display_name", "season", "week", "position", "fantasy_points_ppr"] + feature_cols].head())

## 8. Visualize relationship between recent points and next points

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=model_df, x="prior3_fantasy_points_ppr", y="fantasy_points_ppr", hue="position", alpha=0.4)
plt.title("Previous 3-Game Average vs Current-Week PPR Points")
plt.xlabel("Previous 3 Appearances: Average PPR")
plt.ylabel("Current-Week Actual PPR")
plt.tight_layout()
plt.show()

correlation = model_df["prior3_fantasy_points_ppr"].corr(model_df["fantasy_points_ppr"])
print(f"Correlation: {correlation:.3f}")

## 9. Train/test split without future-data leakage

Use 2023–2024 to train; evaluate on 2025. Don't shuffle player-weeks across time, since that can make a predictive model appear more accurate than it really is. Position is one-hot encoded by the pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train = model_df[model_df["season"].isin([2023, 2024])].copy()
test = model_df[model_df["season"] == 2025].copy()
assert not train.empty and not test.empty, "Training/test data missing — check available seasons."
features = feature_cols + ["position"]
X_train, y_train = train[features], train["fantasy_points_ppr"]
X_test, y_test = test[features], test["fantasy_points_ppr"]

preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), feature_cols),
    ("position", OneHotEncoder(handle_unknown="ignore"), ["position"])
])
linear_model = Pipeline([
    ("prep", preprocessor), ("model", LinearRegression())
])
forest_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=5,
                                     random_state=42, n_jobs=-1))
])
linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

predictions = {
    "Prior 3-game average (baseline)": test["prior3_fantasy_points_ppr"].to_numpy(),
    "Linear regression": linear_model.predict(X_test),
    "Random forest": forest_model.predict(X_test)
}
results = []
for name, pred in predictions.items():
    results.append({"Model": name,
                    "MAE": mean_absolute_error(y_test, pred),
                    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
                    "R2": r2_score(y_test, pred)})
results_df = pd.DataFrame(results).sort_values("MAE")
display(results_df.round(3))
print("MAE tells us typical prediction error in PPR points (smaller is better).")

## 10. Visualize model errors

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="Model", y="MAE")
plt.title("2025 Holdout Test: Mean Absolute Prediction Error")
plt.xlabel("Model")
plt.ylabel("MAE (Fantasy PPR Points)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

best_name = results_df.iloc[0]["Model"]
best_preds = predictions[best_name]
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_preds, alpha=0.4)
limits = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(limits, limits, linestyle="--", color="black", label="Perfect predictions")
plt.xlabel("Actual 2025 PPR points")
plt.ylabel("Predicted 2025 PPR points")
plt.title(f"Actual vs Predicted: {best_name}")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Inspect predictions and export your cleaned dataset

In [ ]:
pred_table = test[["season", "week", "player_display_name", "position", "fantasy_points_ppr"]].copy()
pred_table["predicted_ppr"] = best_preds
pred_table["absolute_error"] = abs(pred_table["fantasy_points_ppr"] - pred_table["predicted_ppr"])
display(pred_table.head(15).round(2))

model_df.to_csv("fantasy_football_model_data.csv", index=False)
pred_table.to_csv("fantasy_football_2025_predictions.csv", index=False)
print("Saved fantasy_football_model_data.csv and fantasy_football_2025_predictions.csv")

## 12. Kickoff worksheet: knowledge gaps and next steps

### What we know
- NFLverse contains weekly player statistics with PPR scoring fields.
- Regression can predict a continuous quantity such as fantasy points.

### What we think but need to test
- Prior three-game performance may contain predictive information.
- The random forest may beat a simple historical average; the 2025 test determines whether that is true.

### What we need to learn
- Are results different for QB, RB, WR, and TE?
- Do pregame matchup quality, injuries, snaps, or targets improve prediction accuracy?
- Does accuracy improve with additional seasons or different rolling windows?

### Suggested team tasks before next meeting
1. **Data lead:** confirm the fields, missingness, and data quality.
2. **Modeling lead:** run and compare baseline, linear regression, and random forest.
3. **Research/writing lead:** find two background studies, document limitations, and draft research justification.

### Interpretation questions for your final project
- Which model has the lowest holdout MAE? By how many PPR points?
- Which position is easiest/hardest to predict?
- How can the model be improved **without using information known only after kickoff**?
- Who benefits from the model and who could be misled by an inaccurate prediction?

**Note:** The group agreement and brainstorming responses are team decisions; add those after discussion instead of presenting invented commitments as agreed upon.